In [1]:
pip install selenium beautifulsoup4 matplotlib


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from time import sleep
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from tkinter import Tk, Button, Frame, Label
import tkinter.font as tkFont

# ---------- 資料擷取函式 ----------
def scrape_articles():
    print("🔍 開始爬取 NBA 資訊...")
    driver = webdriver.Chrome()
    url = 'https://sports.sina.com.cn/nba/?from=wap'
    driver.get(url)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, 'feed-card-item'))
        )
    except TimeoutException:
        print("⚠️ Timed out waiting for page to load")

    titles = []

    for _ in range(3):  # 爬前 3 頁
        sleep(2)
        articles = driver.find_elements(By.CLASS_NAME, 'feed-card-item')
        for article in articles:
            try:
                title = article.find_element(By.TAG_NAME, 'a').text
                titles.append(title)
            except:
                continue
        try:
            next_page_url = driver.find_element(By.CSS_SELECTOR, '.pagination-next').get_attribute('href')
            driver.get(next_page_url)
        except:
            break

    driver.quit()
    print(f"✅ 共擷取 {len(titles)} 筆文章資料。")
    return titles

# ---------- 分析與畫圖函式 ----------
def analyze_and_plot(titles):
    players_chinese = ['约基奇', '莫兰特', '哈利伯顿', '杜兰特', '塔图姆', '亚历山大', '恩比德', '詹姆斯', '库里']
    players_english = ['Jokic', 'Morant', 'Haliburton', 'Durant', 'Tatum', 'SGA', 'Embiid', 'James', 'Curry']
    colors = ['#427f8f', '#4a8fa1', '#559db0', '#66a7b8', '#77b1c0', '#89bbc8', '#9ac5d0', '#bdd9e0', '#cee3e8']

    counts = [sum(name in title for title in titles) for name in players_chinese]

    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(range(len(players_english)), counts, color=colors)

    ax.set_xticks(range(len(players_english)))
    ax.set_xticklabels(players_english, rotation=45, ha='right')
    ax.set_xlabel('Player', fontsize=14)
    ax.set_ylabel('Mention Count', fontsize=14)
    ax.set_title('NBA Player Volume Analysis', fontsize=18)

    for i, bar in enumerate(bars):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(counts[i]),
                ha='center', va='bottom', fontsize=10)

    return fig

# ---------- 畫圖按鈕功能 ----------
def plot_graph():
    titles = scrape_articles()
    fig = analyze_and_plot(titles)

    for widget in plot_frame.winfo_children():
        widget.destroy()

    canvas = FigureCanvasTkAgg(fig, master=plot_frame)
    canvas.draw()
    canvas.get_tk_widget().pack(pady=10)

# ---------- Tkinter GUI ----------
root = Tk()
root.geometry('850x900')
root.title("NBA 熱門球員文章分析")

header_font = tkFont.Font(family="Helvetica", size=18, weight="bold")

# 標題
header_label = Label(root, text="🏀 NBA 熱門球員聲量分析", font=header_font, pady=20)
header_label.pack()

# 按鈕
plot_button = Button(root, text="點擊爬取並分析文章", command=plot_graph,
                     font=('Helvetica', 14), width=30, height=2, bg="#3182bd", fg="white")
plot_button.pack(pady=10)

# 圖表區域（放最下方）
plot_frame = Frame(root)
plot_frame.pack(pady=20)

# 啟動視窗
root.mainloop()
plt.show

🔍 開始爬取 NBA 資訊...
✅ 共擷取 30 筆文章資料。
